# LAK-11 — Hudi's timeline & copy-on-write upserts

Apache Hudi, like Iceberg and Delta, is a **table format** over Parquet on object storage. What makes it *Hudi* is two things you'll see directly in this lesson:

1. **The timeline** — every write appends an *instant* under a `.hoodie/` directory. That timeline is Hudi's source of truth (Iceberg uses a metadata/manifest tree; Delta uses a `_delta_log/`).
2. **Record-level upserts** — Hudi tables declare a **record key** (`primaryKey`) and a **precombine field**, so `MERGE`/upsert is first-class rather than bolted on.

**What we'll do (Break → Detect → Prove):** declare a copy-on-write (CoW) Hudi table, seed it, read the raw timeline, upsert one row with `MERGE`, then watch a *new* commit instant + a rewritten base file appear — while the row count stays put.

> Prereqs: `make up` (MiniStack + Spark Connect). Data lands in `s3://warehouse/hudi/`.

In [1]:
from common.spark_session import spark
# Thin, boilerplate-only helpers: an S3 client, a URI splitter, a prefix wipe,
# and a health probe. The Hudi *technique* (reading the timeline) is shown
# inline below — not hidden in a helper.
from common.table_meta import hudi_table_health as table_health, wipe_prefix, s3_client, split_s3

TABLE = "spark_catalog.default.lak11_orders"
LOCATION = "s3a://warehouse/hudi/lak11/orders"

# Clean start. A Hudi DROP TABLE removes the catalog entry but NOT the S3 files,
# so we clear the prefix directly (idempotent — safe to re-run this notebook).
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")
removed = wipe_prefix(LOCATION)
print(f"reset: dropped {TABLE} + wiped {removed} object(s) under {LOCATION}")

reset: dropped spark_catalog.default.lak11_orders + wiped 55 object(s) under s3a://warehouse/hudi/lak11/orders


## 1. Declare a Hudi table — the DDL that actually matters

The distinctive part of a Hudi `CREATE TABLE` is the `TBLPROPERTIES`:

| property | meaning |
|---|---|
| `primaryKey` | the **record key** — how Hudi identifies a row across upserts |
| `preCombineField` | tie-breaker when two records share a key (latest `updated_at` wins) |
| `type` | `cow` (copy-on-write) or `mor` (merge-on-read) — see LAK-12 |

Write this DDL by hand — this is exactly what you'd author in a real job. Nothing here is lab-specific except the `LOCATION`.

In [2]:
spark.sql(f"""
CREATE TABLE {TABLE} (
    order_id    BIGINT,
    customer_id BIGINT,
    amount      DOUBLE,
    status      STRING,
    updated_at  TIMESTAMP
) USING hudi
LOCATION '{LOCATION}'
TBLPROPERTIES (
    'primaryKey'      = 'order_id',
    'preCombineField' = 'updated_at',
    'type'            = 'cow'
)
""")
print("created CoW Hudi table")

created CoW Hudi table


## 2. Seed three rows

Each write appends an instant to the timeline. We capture a `before` health snapshot right after the insert so we can compare it to the post-upsert state.

In [3]:
spark.sql(f"""
INSERT INTO {TABLE} VALUES
    (1, 100, 50.0,  'NEW',  current_timestamp()),
    (2, 101, 75.0,  'PAID', current_timestamp()),
    (3, 102, 120.0, 'NEW',  current_timestamp())
""")

before = table_health(LOCATION)
print("BEFORE upsert:", before)

BEFORE upsert: {'latest_commit': '20260721115032030_20260721115033305', 'parquet_files_on_disk': 1, 'log_files': 0, 'total_bytes': 435754}


## 3. Read the timeline directly (the Hudi-specific bit)

The timeline lives in `<table>/.hoodie/`. Each **instant** is a file named `<timestamp>.<action>`:

- `.commit` — a copy-on-write write completed
- `.deltacommit` — a merge-on-read write completed (log files)
- `.replacecommit` — clustering / `INSERT OVERWRITE`
- `.requested` / `.inflight` — in-progress state markers for an instant

Let's list it ourselves instead of trusting a helper — this is the artifact the whole format is built around.

In [4]:
s3 = s3_client()
bucket, prefix = split_s3(LOCATION)

def show_timeline(label):
    """List the instant files under .hoodie/ (skip the internal metadata/ dir)."""
    print(f"\n.hoodie/ timeline — {label}:")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/.hoodie/")
    names = sorted(o["Key"].rsplit("/", 1)[-1] for o in resp.get("Contents", []))
    for n in names:
        # instants start with a numeric timestamp; also show hoodie.properties
        if n and (n[0].isdigit() or n == "hoodie.properties"):
            print(f"   {n}")

show_timeline("after initial insert")


.hoodie/ timeline — after initial insert:
   00000000000000000.deltacommit.inflight
   00000000000000000.deltacommit.requested
   00000000000000000_20260721115032355.deltacommit
   00000000000000001.deltacommit.inflight
   00000000000000001.deltacommit.requested
   00000000000000001_20260721115032588.deltacommit
   20260721115032030.commit.requested
   20260721115032030.deltacommit.inflight
   20260721115032030.deltacommit.requested
   20260721115032030.inflight
   20260721115032030_20260721115033251.deltacommit
   20260721115032030_20260721115033305.commit
   hoodie.properties
   hoodie.properties


## 4. Upsert one row with `MERGE` (copy-on-write)

We ship order #1 (`NEW` → `SHIPPED`). Because this table is **copy-on-write**, Hudi rewrites the base Parquet file(s) that hold the changed record and writes a *new* `.commit` instant.

> Contrast (LAK-12): a **merge-on-read** table would instead append a compact **log file** (`.log.`) and defer the rewrite to compaction — cheaper writes, more work on read.

In [5]:
spark.sql(f"""
MERGE INTO {TABLE} t
USING (
    SELECT CAST(1 AS BIGINT)   AS order_id,
           CAST(100 AS BIGINT) AS customer_id,
           50.0                AS amount,
           'SHIPPED'           AS status,
           current_timestamp() AS updated_at
) s
ON t.order_id = s.order_id
WHEN MATCHED     THEN UPDATE SET status = s.status, updated_at = s.updated_at
WHEN NOT MATCHED THEN INSERT *
""")

after = table_health(LOCATION)
print("AFTER upsert: ", after)

AFTER upsert:  {'latest_commit': '20260721115034183_20260721115035368', 'parquet_files_on_disk': 2, 'log_files': 0, 'total_bytes': 871456}


In [6]:
# A second .commit instant should now sit alongside the first.
show_timeline("after MERGE")


.hoodie/ timeline — after MERGE:
   00000000000000000.deltacommit.inflight
   00000000000000000.deltacommit.requested
   00000000000000000_20260721115032355.deltacommit
   00000000000000001.deltacommit.inflight
   00000000000000001.deltacommit.requested
   00000000000000001_20260721115032588.deltacommit
   20260721115032030.commit.requested
   20260721115032030.deltacommit.inflight
   20260721115032030.deltacommit.requested
   20260721115032030.inflight
   20260721115032030_20260721115033251.deltacommit
   20260721115032030_20260721115033305.commit
   20260721115034183.commit.requested
   20260721115034183.deltacommit.inflight
   20260721115034183.deltacommit.requested
   20260721115034183.inflight
   20260721115034183_20260721115035318.deltacommit
   20260721115034183_20260721115035368.commit
   hoodie.properties
   hoodie.properties


## 5. Prove it — one row changed, a new base file was written, bytes grew

Two honest points a lot of Hudi intros get wrong:

- **Row count is unchanged** (still 3) — the upsert *replaced* a row, it didn't append one.
- **`parquet_files_on_disk` may read as 2, but only one is live.** Copy-on-write wrote a fresh base slice; the old slice lingers until Hudi's **cleaner** reclaims it. So we don't claim "storage doubled" — we look at **bytes**, which is the honest measure of write amplification.

In [7]:
rows = spark.sql(f"SELECT order_id, status FROM {TABLE} ORDER BY order_id").collect()
status_by_id = {r["order_id"]: r["status"] for r in rows}
print("rows:", [r.asDict() for r in rows])

assert len(rows) == 3, f"expected 3 rows, got {len(rows)}"
assert status_by_id[1] == "SHIPPED", "order 1 should be SHIPPED after the upsert"

print()
print(f"parquet files on disk : {before['parquet_files_on_disk']} → {after['parquet_files_on_disk']}"
      "  (extra slice is stale until the cleaner runs)")
print(f"bytes on disk         : {before['total_bytes']} → {after['total_bytes']}"
      "  (← the real copy-on-write cost)")
print(f"latest commit instant : {before['latest_commit']} → {after['latest_commit']}")
print("\nLAK-11 OK — CoW upsert rewrote a base file and advanced the timeline.")

rows: [{'order_id': 1, 'status': 'SHIPPED'}, {'order_id': 2, 'status': 'PAID'}, {'order_id': 3, 'status': 'NEW'}]

parquet files on disk : 1 → 2  (extra slice is stale until the cleaner runs)
bytes on disk         : 435754 → 871456  (← the real copy-on-write cost)
latest commit instant : 20260721115032030_20260721115033305 → 20260721115034183_20260721115035368

LAK-11 OK — CoW upsert rewrote a base file and advanced the timeline.


## What you just saw

- A Hudi table is declared with a **record key** + **precombine** field; `MERGE` upserts are first-class.
- Every write appends an **instant** to the `.hoodie/` **timeline** — you read it directly, the same way you'd debug a real Hudi table.
- **Copy-on-write** rewrites base Parquet on every upsert; the write cost shows up in **bytes**, not (misleadingly) in file *count*.

### Try it yourself
- Re-run the `MERGE` a few times and watch new `.commit` instants stack up.
- Add `CALL run_clean(...)` (Hudi's cleaner) and re-check `parquet_files_on_disk` — the stale slices go away.
- Change `'type'='cow'` → `'mor'` and watch `.log.` files appear instead of base rewrites.

**Next:** [LAK-12 — CoW vs MoR across Iceberg & Hudi](./lak12_cow_vs_mor.ipynb) quantifies the write-amplification trade-off.